In [1]:
import cv2
import numpy as np
import os
from skimage.feature import hog
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
import joblib
import os
from tqdm import tqdm
import time
import pandas as pd
from sklearn.metrics import roc_curve, confusion_matrix
from xgboost import XGBClassifier
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import numpy as np


In [2]:
def load_images(folder, label):
    images = []
    labels = []
    files = [f for f in os.listdir(folder) if f.endswith('.png') or f.endswith('.jpg')]
    for filename in files:
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = cv2.resize(img, (128, 128))
        images.append(img)
        labels.append(label)
    return images, labels

healthy_path = "/Users/nabiha/parkinsons_drawings/Healthy"
parkinson_path = "/Users/nabiha/parkinsons_drawings/Parkinson"

healthy_imgs, healthy_labels = load_images(healthy_path, 0)
parkinson_imgs, parkinson_labels = load_images(parkinson_path, 1)

all_images = healthy_imgs + parkinson_imgs
all_labels = healthy_labels + parkinson_labels

print("Total images:", len(all_images))
print("Class distribution — Healthy:", len(healthy_imgs), "Parkinson:", len(parkinson_imgs))

Total images: 3389
Class distribution — Healthy: 1757 Parkinson: 1632


In [3]:
def extract_hog_features(images):
    features = []
    for img in images:
        fd = hog(img, 
                 orientations=9, 
                 pixels_per_cell=(8, 8),
                 cells_per_block=(2, 2), 
                 block_norm='L2-Hys')
        features.append(fd)
    return np.array(features)

X = extract_hog_features(all_images)
y = np.array(all_labels)

print("Feature matrix shape:", X.shape)

Feature matrix shape: (3389, 8100)


In [4]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (2711, 8100)
X_test shape: (678, 8100)


In [5]:
pca = PCA(n_components=100, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("X_train_pca shape:", X_train_pca.shape)
print("X_test_pca shape:", X_test_pca.shape)
print("Variance explained:", round(sum(pca.explained_variance_ratio_) * 100, 2), "%")

X_train_pca shape: (2711, 100)
X_test_pca shape: (678, 100)
Variance explained: 57.43 %


In [6]:
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("Number of components selected:", pca.n_components_)
print("X_train_pca shape:", X_train_pca.shape)
print("Variance explained:", round(sum(pca.explained_variance_ratio_) * 100, 2), "%")

Number of components selected: 1055
X_train_pca shape: (2711, 1055)
Variance explained: 95.01 %


In [7]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10],
    "min_samples_split": [2, 5]
}

rf = RandomForestClassifier(class_weight='balanced', random_state=42)

grid_search = GridSearchCV(
    estimator=rf, 
    param_grid=param_grid, 
    scoring='roc_auc', 
    cv=5,
    verbose=2
)

grid_search.fit(X_train_pca, y_train)

print("Best params:", grid_search.best_params_)
print("Best ROC-AUC:", grid_search.best_score_)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   3.2s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   3.3s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   3.3s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   3.2s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   3.2s
[CV] END max_depth=None, min_samples_split=2, n_estimators=200; total time=   6.7s
[CV] END max_depth=None, min_samples_split=2, n_estimators=200; total time=   6.6s
[CV] END max_depth=None, min_samples_split=2, n_estimators=200; total time=   6.6s
[CV] END max_depth=None, min_samples_split=2, n_estimators=200; total time=   6.4s
[CV] END max_depth=None, min_samples_split=2, n_estimators=200; total time=   6.9s
[CV] END max_depth=None, min_samples_split=5, n_estimators=100; total time=   3.7s
[CV] END max_depth=None, mi

In [8]:
best_drawing_rf = grid_search.best_estimator_

y_pred = best_drawing_rf.predict(X_test_pca)
y_prob = best_drawing_rf.predict_proba(X_test_pca)[:, 1]

print("Drawing Model Results")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("ROC AUC:", round(roc_auc_score(y_test, y_prob), 3))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Drawing Model Results
Accuracy: 0.897

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.98      0.91       352
           1       0.98      0.80      0.88       326

    accuracy                           0.90       678
   macro avg       0.91      0.89      0.90       678
weighted avg       0.91      0.90      0.90       678

ROC AUC: 0.973

Confusion Matrix:
[[346   6]
 [ 64 262]]


In [9]:
joblib.dump(best_drawing_rf, '../models/drawing_model.pkl')
joblib.dump(pca, '../models/drawing_pca.pkl')

['../models/drawing_pca.pkl']

In [10]:
def fuse_predictions(clinical_prob, drawing_prob):
    diff = abs(clinical_prob - drawing_prob)
    
    if clinical_prob >= 0.65 and drawing_prob >= 0.65:
        return "High Risk — both models agree"
    elif clinical_prob <= 0.35 and drawing_prob <= 0.35:
        return "Low Risk — both models agree"
    elif diff > 0.3:
        return "Inconclusive — models disagree"
    else:
        return "Moderate Risk"

In [11]:
models_dir = '../models/'
print("Files in models folder:")
for f in sorted(os.listdir(models_dir)):
    print(f"  {f}")


Files in models folder:
  X_test_scaled.pkl
  X_train_scaled.pkl
  class_distribution.csv
  clinical_model.pkl
  clinical_scaler.pkl
  confusion_matrix.csv
  confusion_matrix_drawing.csv
  drawing_cnn.pth
  drawing_comparison.csv
  drawing_model.pkl
  drawing_pca.pkl
  feature_importance.csv
  model_comparison.csv
  roc_clinical.csv
  roc_drawing.csv
  selected_features.pkl
  shap_explainer.pkl
  shap_values.pkl
  y_test.pkl
  y_train.pkl


In [12]:
clinical_model = joblib.load('../models/clinical_model.pkl')
clinical_scaler = joblib.load('../models/clinical_scaler.pkl')
selected_features = joblib.load('../models/selected_features.pkl')
drawing_model = joblib.load('../models/drawing_model.pkl')
drawing_pca = joblib.load('../models/drawing_pca.pkl')
shap_explainer = joblib.load('../models/shap_explainer.pkl')

print("All models loaded successfully!")
print(f"Clinical model: {type(clinical_model).__name__}")
print(f"Drawing model: {type(drawing_model).__name__}")
print(f"Selected features ({len(selected_features)}):", selected_features)

All models loaded successfully!
Clinical model: RandomForestClassifier
Drawing model: RandomForestClassifier
Selected features (13): ['Age', 'DietQuality', 'TraumaticBrainInjury', 'Diabetes', 'Depression', 'UPDRS', 'MoCA', 'FunctionalAssessment', 'Tremor', 'Rigidity', 'Bradykinesia', 'PosturalInstability', 'SleepDisorders']


/opt/anaconda3/envs/parkinsons_jcp/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:

fusion_code = '''

def fuse_predictions(clinical_prob, drawing_prob):
    diff = abs(clinical_prob - drawing_prob)
    
    if clinical_prob >= 0.65 and drawing_prob >= 0.65:
        return "High Risk — both models agree"
    elif clinical_prob <= 0.35 and drawing_prob <= 0.35:
        return "Low Risk — both models agree"
    elif diff > 0.3:
        return "Inconclusive — models disagree"
    else:
        return "Moderate Risk"
'''

with open('../fusion.py', 'w') as f:
    f.write(fusion_code)

In [14]:

# ROC data for drawing model
fpr_drawing, tpr_drawing, _ = roc_curve(y_test, y_prob)
roc_drawing = pd.DataFrame({'FPR': fpr_drawing, 'TPR': tpr_drawing})
roc_drawing.to_csv('../models/roc_drawing.csv', index=False)

# Confusion matrix for drawing
cm_drawing = confusion_matrix(y_test, y_pred)
cm_drawing_df = pd.DataFrame(cm_drawing,
                              index=['Actual Healthy', 'Actual PD'],
                              columns=['Predicted Healthy', 'Predicted PD'])
cm_drawing_df.to_csv('../models/confusion_matrix_drawing.csv')

In [15]:
# Calculate class weight for imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# XGBoost model
xgb = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='auc'
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 6],
    "learning_rate": [0.05, 0.1]
}

grid_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=5,
    verbose=1
)

grid_xgb.fit(X_train_pca, y_train)

print("Best params:", grid_xgb.best_params_)
print("Best ROC-AUC:", grid_xgb.best_score_)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200}
Best ROC-AUC: 0.9583087097799247


In [16]:
best_xgb = grid_xgb.best_estimator_

y_pred_xgb = best_xgb.predict(X_test_pca)
y_prob_xgb = best_xgb.predict_proba(X_test_pca)[:, 1]

print("XGBoost Drawing Model Results")
print("Accuracy:", round(accuracy_score(y_test, y_pred_xgb), 3))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))
print("ROC AUC:", round(roc_auc_score(y_test, y_prob_xgb), 3))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

XGBoost Drawing Model Results
Accuracy: 0.91

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.96      0.92       352
           1       0.95      0.86      0.90       326

    accuracy                           0.91       678
   macro avg       0.92      0.91      0.91       678
weighted avg       0.91      0.91      0.91       678

ROC AUC: 0.977

Confusion Matrix:
[[338  14]
 [ 47 279]]


In [17]:
joblib.dump(best_xgb, '../models/drawing_model.pkl')

['../models/drawing_model.pkl']

In [18]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# Image transformations — MobileNetV2 expects 224x224 RGB normalized
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),  # convert grayscale to 3-channel
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# Custom dataset class
class SpiralDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

Using device: mps


In [19]:
import os

healthy_dir = "/Users/nabiha/parkinsons_drawings/Healthy"
parkinson_dir = "/Users/nabiha/parkinsons_drawings/Parkinson"

healthy_paths = [os.path.join(healthy_dir, f) for f in os.listdir(healthy_dir) if f.endswith('.png')]
parkinson_paths = [os.path.join(parkinson_dir, f) for f in os.listdir(parkinson_dir) if f.endswith('.png')]

all_paths = healthy_paths + parkinson_paths
all_labels_cnn = [0] * len(healthy_paths) + [1] * len(parkinson_paths)

print(f"Total images: {len(all_paths)}")
print(f"Healthy: {len(healthy_paths)}, Parkinson: {len(parkinson_paths)}")

# Train/test split
from sklearn.model_selection import train_test_split
train_paths, test_paths, train_labels, test_labels = train_test_split(
    all_paths, all_labels_cnn, test_size=0.2, random_state=42, stratify=all_labels_cnn
)

# Create datasets and loaders
train_dataset = SpiralDataset(train_paths, train_labels, transform=transform)
test_dataset = SpiralDataset(test_paths, test_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Training batches: {len(train_loader)}, Test batches: {len(test_loader)}")

Total images: 3389
Healthy: 1757, Parkinson: 1632
Training batches: 85, Test batches: 22


In [20]:
# Load pretrained MobileNetV2
model = models.mobilenet_v2(weights='IMAGENET1K_V1')

# Freeze all pretrained layers (we don't want to retrain ImageNet features)
for param in model.parameters():
    param.requires_grad = False

# Replace the final classification layer for binary classification
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)

# Move to GPU
model = model.to(device)

print("Model ready")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Model ready
Trainable parameters: 2,562


In [21]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{num_epochs} — Loss: {running_loss/len(train_loader):.4f} — Accuracy: {train_acc:.2f}%")

Epoch 1/5 — Loss: 0.5065 — Accuracy: 75.88%
Epoch 2/5 — Loss: 0.4036 — Accuracy: 82.22%
Epoch 3/5 — Loss: 0.3643 — Accuracy: 83.84%
Epoch 4/5 — Loss: 0.3389 — Accuracy: 85.84%
Epoch 5/5 — Loss: 0.3328 — Accuracy: 85.65%


In [22]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

model.eval()
all_preds = []
all_probs = []
all_labels_test = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels_test.extend(labels.numpy())

print("CNN Test Results")
print("Accuracy:", round(accuracy_score(all_labels_test, all_preds), 3))
print("\nClassification Report:")
print(classification_report(all_labels_test, all_preds))
print("ROC AUC:", round(roc_auc_score(all_labels_test, all_probs), 3))
print("\nConfusion Matrix:")
print(confusion_matrix(all_labels_test, all_preds))

CNN Test Results
Accuracy: 0.894

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.88      0.90       352
           1       0.88      0.90      0.89       326

    accuracy                           0.89       678
   macro avg       0.89      0.89      0.89       678
weighted avg       0.89      0.89      0.89       678

ROC AUC: 0.956

Confusion Matrix:
[[311  41]
 [ 31 295]]


In [23]:
torch.save(model.state_dict(), '../models/drawing_cnn.pth')

In [24]:
torch.save(model.state_dict(), '../models/drawing_cnn.pth')
print("CNN model saved!")

CNN model saved!


In [25]:
cm_cnn = confusion_matrix(all_labels_test, all_preds)
cm_cnn_df = pd.DataFrame(cm_cnn,
                          index=['Actual Healthy', 'Actual PD'],
                          columns=['Predicted Healthy', 'Predicted PD'])
cm_cnn_df.to_csv('../models/confusion_matrix_drawing.csv')

In [26]:
import torch.optim as optim

# Unfreeze the last 30 layers
for param in list(model.parameters())[-30:]:
    param.requires_grad = True

# New optimizer with lower learning rate
optimizer = optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.0001
)

# Train 10 more epochs
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{num_epochs} — Loss: {running_loss/len(train_loader):.4f} — Accuracy: {train_acc:.2f}%")

Epoch 1/5 — Loss: 0.3012 — Accuracy: 87.83%
Epoch 2/5 — Loss: 0.1441 — Accuracy: 94.84%
Epoch 3/5 — Loss: 0.1085 — Accuracy: 95.94%
Epoch 4/5 — Loss: 0.0800 — Accuracy: 97.09%
Epoch 5/5 — Loss: 0.0525 — Accuracy: 98.08%


In [27]:
model.eval()
all_preds = []
all_probs = []
all_labels_test = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels_test.extend(labels.numpy())

print("Fine-tuned CNN Test Results")
print("Accuracy:", round(accuracy_score(all_labels_test, all_preds), 3))
print("\nClassification Report:")
print(classification_report(all_labels_test, all_preds))
print("ROC AUC:", round(roc_auc_score(all_labels_test, all_probs), 3))
print("\nConfusion Matrix:")
print(confusion_matrix(all_labels_test, all_preds))

Fine-tuned CNN Test Results
Accuracy: 0.973

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.99      0.97       352
           1       0.98      0.96      0.97       326

    accuracy                           0.97       678
   macro avg       0.97      0.97      0.97       678
weighted avg       0.97      0.97      0.97       678

ROC AUC: 0.995

Confusion Matrix:
[[347   5]
 [ 13 313]]


In [28]:
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
import pandas as pd

In [31]:
torch.save(model.state_dict(), '../models/drawing_cnn.pth')

from sklearn.metrics import confusion_matrix, roc_curve, accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

cm_cnn = confusion_matrix(all_labels_test, all_preds)
cm_cnn_df = pd.DataFrame(cm_cnn,
                          index=['Actual Healthy', 'Actual PD'],
                          columns=['Predicted Healthy', 'Predicted PD'])
cm_cnn_df.to_csv('../models/confusion_matrix_drawing.csv')

drawing_comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'ROC AUC', 'Precision', 'Recall', 'F1 Score'],
    'XGBoost (HOG+PCA)': [
        accuracy_score(y_test, y_pred_xgb),
        roc_auc_score(y_test, y_prob_xgb),
        precision_score(y_test, y_pred_xgb),
        recall_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_xgb)
    ],
    'CNN (MobileNetV2)': [
        accuracy_score(all_labels_test, all_preds),
        roc_auc_score(all_labels_test, all_probs),
        precision_score(all_labels_test, all_preds),
        recall_score(all_labels_test, all_preds),
        f1_score(all_labels_test, all_preds)
    ]
})
drawing_comparison.to_csv('../models/drawing_comparison.csv', index=False)

fpr_drawing, tpr_drawing, _ = roc_curve(all_labels_test, all_probs)
roc_drawing = pd.DataFrame({'FPR': fpr_drawing, 'TPR': tpr_drawing})
roc_drawing.to_csv('../models/roc_drawing.csv', index=False)